# 02 — RAG Exploration (Greek Mythology)

Run from the **repository root** after installing `requirements-rag.txt`. This notebook explains **retrieval**, not LLM model training. The first embedding-model call downloads weights from Hugging Face.


In [ ]:
from src.rag.knowledge_base import load_documents
from src.rag.text_chunker import chunk_documents

documents = load_documents()
chunks = chunk_documents(documents)
print(f"Documents: {len(documents)} | Chunks: {len(chunks)}")
print("First source:", documents[0])


## Chunking experiment

The current 63 summaries are short, so each is one chunk at default settings. Reduce `max_words` to simulate longer source documents.


In [ ]:
short_chunks = chunk_documents(documents[:2], max_words=12, overlap_words=3)
for chunk in short_chunks[:4]:
    print(chunk.id, repr(chunk.text))


## Embedding inference + FAISS indexing

`all-MiniLM-L6-v2` maps English passages into numerical vectors. FAISS uses normalized inner-product search, equivalent to cosine similarity for unit vectors. **Scores are not probabilities.**


In [ ]:
from src.rag.pipeline import build_pipeline
store = build_pipeline()
print("Index passages:", store.index.ntotal)
print("Vector dimension:", store.index.d)


In [ ]:
question = "How are Apollo and Artemis related?"
hits = store.search(question, top_k=5, min_score=0.0)
for rank, hit in enumerate(hits, start=1):
    print(f"{rank}. {hit.chunk.title} | similarity={hit.score:.3f}")
    print("  ", hit.chunk.text)
    print("  ", hit.chunk.source_url)


## Evidence-only answer

Without API credentials the system returns retrieved passages, **not** an LLM-generated narrative. In a future milestone we will evaluate no-answer detection and fact-level citation faithfulness.


In [ ]:
from src.rag.answer_service import answer_question
result = answer_question(question, store)
print(result.mode)
print(result.text)


## Tiny retrieval evaluation

The included benchmark is hand-authored and very small: suitable as a regression check, not as a validated accuracy claim.


In [ ]:
from src.rag.evaluate import evaluate_retrieval
from pathlib import Path
import json

examples = json.loads(Path("data/evaluation/retrieval_eval.json").read_text(encoding="utf-8"))
print(evaluate_retrieval(store, examples))
